In [1]:
%%time
# =============================================================================
# Redrob Ranker — Step 3
# Availability Multiplier
# Jupyter single-cell version
#
# Inputs:
#   - survivors.parquet              candidate_id list from Steps 1–2
#   - candidate_base.parquet         profile fields, including country
#   - candidate_redrob.parquet       behavioral/platform signals
#
# Output:
#   - availability_scores.parquet
#
# Design:
#   availability_multiplier =
#       country_gate × recency_gate × engagement_score
#
#   engagement_score =
#       response_score  × 0.28
#     + notice_score    × 0.30
#     + intent_score    × 0.18
#     + interview_score × 0.12
#     + work_mode_score × 0.12
#
# Notes:
#   - City/location and willing_to_relocate are intentionally excluded.
#   - JD says Pune/Noida preferred but flexible, no fixed office cadence,
#     quarterly travel/offsites, and outside India is case-by-case with no visa sponsorship.
#   - Step 3 is a soft filter, not a hard drop.
# =============================================================================

import polars as pl
from datetime import date
from pathlib import Path
from IPython.display import display


# ── Constants ────────────────────────────────────────────────────────────────

TODAY = date(2026, 6, 25)

FLOOR = 0.05

RATE_FLOOR = 0.10
RATE_CEIL  = 0.90

# Engagement weights — observed behavior > self-reported checkbox
W_RESPONSE  = 0.28
W_NOTICE    = 0.30
W_INTENT    = 0.18
W_INTERVIEW = 0.12
W_WORK_MODE = 0.12

assert abs(
    W_RESPONSE + W_NOTICE + W_INTENT + W_INTERVIEW + W_WORK_MODE - 1.0
) < 1e-9, "Engagement weights must sum to 1.00"


OUTPUT_COLS = [
    "candidate_id",

    # Final result
    "availability_multiplier",
    "availability_raw",

    # Multiplicative gates
    "country_gate",
    "recency_gate",

    # Engagement
    "engagement_score",
    "response_score",
    "notice_score",
    "intent_score",
    "interview_score",
    "work_mode_score",

    # Debug / explainability helpers
    "last_active_days_ago",
    "response_rate_flag",
    "country_risk_flag",
    "stale_activity_flag",
    "long_notice_flag",
    "not_open_flag",
    "weak_interview_flag",
    "remote_pref_flag",
]


# ── Path resolver ────────────────────────────────────────────────────────────
# This makes the cell work whether your parquets are in:
#   artifacts/
#   dataset/artifacts/
#   dataset/

def find_file(*candidates: str) -> Path:
    for c in candidates:
        p = Path(c)
        if p.exists():
            return p
    raise FileNotFoundError(
        "Could not find any of these files:\n" +
        "\n".join(f"  - {c}" for c in candidates)
    )


SURVIVORS_PATH = find_file(
    "artifacts/survivors.parquet",
    "dataset/artifacts/survivors.parquet",
)

BASE_PATH = find_file(
    "artifacts/candidate_base.parquet",
    "dataset/artifacts/candidate_base.parquet",
    "dataset/candidate_base.parquet",
)

REDROB_PATH = find_file(
    "artifacts/candidate_redrob.parquet",
    "dataset/artifacts/candidate_redrob.parquet",
    "dataset/candidate_redrob.parquet",
)

OUTPUT_PATH = SURVIVORS_PATH.parent / "availability_scores.parquet"

print("Resolved paths:")
print(f"  survivors : {SURVIVORS_PATH}")
print(f"  base      : {BASE_PATH}")
print(f"  redrob    : {REDROB_PATH}")
print(f"  output    : {OUTPUT_PATH}")


# ── Load & Join ──────────────────────────────────────────────────────────────

def load_data() -> pl.DataFrame:
    survivors = (
        pl.read_parquet(SURVIVORS_PATH)
        .select("candidate_id")
        .unique()
    )

    base = (
        pl.read_parquet(BASE_PATH)
        .select([
            "candidate_id",
            "country",
        ])
    )

    redrob = (
        pl.read_parquet(REDROB_PATH)
        .select([
            "candidate_id",
            "last_active_date",
            "open_to_work_flag",
            "recruiter_response_rate",
            "notice_period_days",
            "preferred_work_mode",
            "interview_completion_rate",
        ])
    )

    df = (
        survivors
        .join(base, on="candidate_id", how="left")
        .join(redrob, on="candidate_id", how="left")
    )

    return df


# ── Gate 1 — Country Gate ────────────────────────────────────────────────────
# City/location and willing_to_relocate are intentionally NOT used.
#
# country == India       → 1.00
# country null/blank     → 0.75
# country != India       → 0.35

def compute_country_gate(df: pl.DataFrame) -> pl.DataFrame:
    df = df.with_columns(
        pl.col("country")
          .cast(pl.Utf8, strict=False)
          .str.to_lowercase()
          .str.strip_chars()
          .alias("_country_norm")
    )

    country_unknown = (
        pl.col("_country_norm").is_null()
        | (pl.col("_country_norm") == "")
    )

    df = df.with_columns(
        pl.when(pl.col("_country_norm") == "india")
          .then(pl.lit(1.00))
          .when(country_unknown)
          .then(pl.lit(0.75))
          .otherwise(pl.lit(0.35))
          .cast(pl.Float64)
          .alias("country_gate")
    )

    return df.drop("_country_norm")


# ── Gate 2 — Recency Gate ────────────────────────────────────────────────────
#
# ≤14 days       → 1.00
# 15–30 days     → 0.95
# 31–60 days     → 0.85
# 61–180 days    → 0.70
# 181–365 days   → 0.45
# >365 days      → 0.20
# null           → 0.30

def compute_recency_gate(df: pl.DataFrame) -> pl.DataFrame:
    today_lit = pl.lit(TODAY)

    df = df.with_columns(
        pl.col("last_active_date")
          .cast(pl.Utf8, strict=False)
          .str.to_date(format="%Y-%m-%d", strict=False)
          .alias("_lad")
    )

    df = df.with_columns(
        pl.when(pl.col("_lad").is_null())
          .then(pl.lit(None, dtype=pl.Int64))
          .otherwise((today_lit - pl.col("_lad")).dt.total_days().cast(pl.Int64))
          .alias("last_active_days_ago")
    )

    days = pl.col("last_active_days_ago")

    df = df.with_columns(
        pl.when(pl.col("_lad").is_null())
          .then(pl.lit(0.30))
          .when(days <= 14)
          .then(pl.lit(1.00))
          .when(days <= 30)
          .then(pl.lit(0.95))
          .when(days <= 60)
          .then(pl.lit(0.85))
          .when(days <= 180)
          .then(pl.lit(0.70))
          .when(days <= 365)
          .then(pl.lit(0.45))
          .otherwise(pl.lit(0.20))
          .cast(pl.Float64)
          .alias("recency_gate")
    )

    return df.drop("_lad")


# ── Engagement Signal 1 — Recruiter Response Rate ────────────────────────────
#
# null              → 0.75
# clamped < 0.20    → 0.30
# 0.20–0.40         → 0.65
# 0.40–0.60         → 0.85
# >0.60             → 1.00

def compute_response_score(df: pl.DataFrame) -> pl.DataFrame:
    df = df.with_columns(
        pl.col("recruiter_response_rate")
          .cast(pl.Float64, strict=False)
          .alias("_rr")
    )

    clamped = pl.col("_rr").clip(RATE_FLOOR, RATE_CEIL)

    df = df.with_columns(
        pl.when(pl.col("_rr").is_null())
          .then(pl.lit(0.75))
          .when(clamped < 0.20)
          .then(pl.lit(0.30))
          .when(clamped < 0.40)
          .then(pl.lit(0.65))
          .when(clamped <= 0.60)
          .then(pl.lit(0.85))
          .otherwise(pl.lit(1.00))
          .cast(pl.Float64)
          .alias("response_score")
    )

    return df.drop("_rr")


# ── Engagement Signal 2 — Notice Period ──────────────────────────────────────
#
# null      → 0.80
# ≤30       → 1.00
# 31–60     → 0.85
# 61–90     → 0.70
# >90       → 0.55

def compute_notice_score(df: pl.DataFrame) -> pl.DataFrame:
    df = df.with_columns(
        pl.col("notice_period_days")
          .cast(pl.Int64, strict=False)
          .alias("_notice_days")
    )

    days = pl.col("_notice_days")

    df = df.with_columns(
        pl.when(days.is_null())
          .then(pl.lit(0.80))
          .when(days <= 30)
          .then(pl.lit(1.00))
          .when(days <= 60)
          .then(pl.lit(0.85))
          .when(days <= 90)
          .then(pl.lit(0.70))
          .otherwise(pl.lit(0.55))
          .cast(pl.Float64)
          .alias("notice_score")
    )

    return df.drop("_notice_days")


# ── Engagement Signal 3 — Intent / Open to Work ──────────────────────────────
#
# true / 1 / "true"       → 1.00
# false / 0 / "false"     → 0.65
# null / blank / unknown  → 0.75

def compute_intent_score(df: pl.DataFrame) -> pl.DataFrame:
    df = df.with_columns(
        pl.col("open_to_work_flag")
          .cast(pl.Utf8, strict=False)
          .str.to_lowercase()
          .str.strip_chars()
          .alias("_otw_str")
    )

    is_true = pl.col("_otw_str").is_in(["true", "1", "yes", "y"])
    is_false = pl.col("_otw_str").is_in(["false", "0", "no", "n"])

    df = df.with_columns(
        pl.when(is_true)
          .then(pl.lit(1.00))
          .when(is_false)
          .then(pl.lit(0.65))
          .otherwise(pl.lit(0.75))
          .cast(pl.Float64)
          .alias("intent_score")
    )

    return df.drop("_otw_str")


# ── Engagement Signal 4 — Interview Completion Rate ─────────────────────────
#
# null              → 0.80
# clamped ≥ 0.80    → 1.00
# 0.50–0.79         → 0.90
# <0.50             → 0.55

def compute_interview_score(df: pl.DataFrame) -> pl.DataFrame:
    df = df.with_columns(
        pl.col("interview_completion_rate")
          .cast(pl.Float64, strict=False)
          .alias("_icr")
    )

    clamped = pl.col("_icr").clip(RATE_FLOOR, RATE_CEIL)

    df = df.with_columns(
        pl.when(pl.col("_icr").is_null())
          .then(pl.lit(0.80))
          .when(clamped >= 0.80)
          .then(pl.lit(1.00))
          .when(clamped >= 0.50)
          .then(pl.lit(0.90))
          .otherwise(pl.lit(0.55))
          .cast(pl.Float64)
          .alias("interview_score")
    )

    return df.drop("_icr")


# ── Engagement Signal 5 — Work Mode ──────────────────────────────────────────
#
# onsite / hybrid / flexible  → 1.00
# remote                      → 0.80
# null / unknown              → 0.90

def compute_work_mode_score(df: pl.DataFrame) -> pl.DataFrame:
    df = df.with_columns(
        pl.col("preferred_work_mode")
          .cast(pl.Utf8, strict=False)
          .str.to_lowercase()
          .str.strip_chars()
          .alias("_wm_norm")
    )

    df = df.with_columns(
        pl.when(pl.col("_wm_norm").is_null() | (pl.col("_wm_norm") == ""))
          .then(pl.lit(0.90))
          .when(pl.col("_wm_norm").is_in(["onsite", "hybrid", "flexible"]))
          .then(pl.lit(1.00))
          .when(pl.col("_wm_norm") == "remote")
          .then(pl.lit(0.80))
          .otherwise(pl.lit(0.90))
          .cast(pl.Float64)
          .alias("work_mode_score")
    )

    return df.drop("_wm_norm")


# ── Engagement Weighted Average ──────────────────────────────────────────────

def compute_engagement_score(df: pl.DataFrame) -> pl.DataFrame:
    return df.with_columns(
        (
            pl.col("response_score")  * W_RESPONSE  +
            pl.col("notice_score")    * W_NOTICE    +
            pl.col("intent_score")    * W_INTENT    +
            pl.col("interview_score") * W_INTERVIEW +
            pl.col("work_mode_score") * W_WORK_MODE
        )
        .cast(pl.Float64)
        .alias("engagement_score")
    )


# ── Combined Multiplier ──────────────────────────────────────────────────────

def compute_multiplier(df: pl.DataFrame) -> pl.DataFrame:
    raw = (
        pl.col("country_gate") *
        pl.col("recency_gate") *
        pl.col("engagement_score")
    )

    return df.with_columns([
        raw.cast(pl.Float64).alias("availability_raw"),
        pl.max_horizontal(raw, pl.lit(FLOOR))
          .cast(pl.Float64)
          .alias("availability_multiplier"),
    ])


# ── Explainability Flags ─────────────────────────────────────────────────────

def compute_flags(df: pl.DataFrame) -> pl.DataFrame:
    rr = pl.col("recruiter_response_rate").cast(pl.Float64, strict=False)
    clamped_rr = rr.clip(RATE_FLOOR, RATE_CEIL)

    notice_days = pl.col("notice_period_days").cast(pl.Int64, strict=False)

    return df.with_columns([
        pl.when(rr.is_null())
          .then(pl.lit(False))
          .when(clamped_rr < 0.40)
          .then(pl.lit(True))
          .otherwise(pl.lit(False))
          .alias("response_rate_flag"),

        (pl.col("country_gate") < 1.00)
          .alias("country_risk_flag"),

        (
            pl.col("last_active_days_ago").is_null()
            | (pl.col("last_active_days_ago") > 180)
        )
          .alias("stale_activity_flag"),

        (
            notice_days.is_not_null()
            & (notice_days > 60)
        )
          .alias("long_notice_flag"),

        (pl.col("intent_score") == 0.65)
          .alias("not_open_flag"),

        (pl.col("interview_score") == 0.55)
          .alias("weak_interview_flag"),

        (pl.col("work_mode_score") == 0.80)
          .alias("remote_pref_flag"),
    ])


# ── Summary ──────────────────────────────────────────────────────────────────

def print_summary(df: pl.DataFrame) -> None:
    total = len(df)

    country_india   = df.filter(pl.col("country_gate") == 1.00).height
    country_unknown = df.filter(pl.col("country_gate") == 0.75).height
    outside_india   = df.filter(pl.col("country_gate") == 0.35).height

    active_14      = df.filter(pl.col("recency_gate") == 1.00).height
    active_15_30   = df.filter(pl.col("recency_gate") == 0.95).height
    active_31_60   = df.filter(pl.col("recency_gate") == 0.85).height
    active_61_180  = df.filter(pl.col("recency_gate") == 0.70).height
    active_181_365 = df.filter(pl.col("recency_gate") == 0.45).height
    active_365p    = df.filter(pl.col("recency_gate") == 0.20).height
    active_null    = df.filter(pl.col("last_active_days_ago").is_null()).height

    low_response   = df.filter(pl.col("response_rate_flag")).height
    long_notice    = df.filter(pl.col("long_notice_flag")).height
    not_open       = df.filter(pl.col("not_open_flag")).height
    weak_interview = df.filter(pl.col("weak_interview_flag")).height
    remote_pref    = df.filter(pl.col("remote_pref_flag")).height

    availability = pl.col("availability_multiplier")

    strong = df.filter(availability >= 0.80).height
    moderate = df.filter((availability >= 0.50) & (availability < 0.80)).height
    weak = df.filter((availability >= 0.20) & (availability < 0.50)).height
    near_floor = df.filter((availability > FLOOR) & (availability < 0.20)).height
    at_floor = df.filter(availability == FLOOR).height
    floor_pct = at_floor / total * 100 if total else 0.0

    print()
    print("=" * 76)
    print("  STEP 3 — AVAILABILITY MULTIPLIER SUMMARY")
    print("=" * 76)
    print(f"  Total processed              : {total:>10,}")
    print()
    print("  Country gate:")
    print(f"    India                      : {country_india:>10,}")
    print(f"    Country unknown/null       : {country_unknown:>10,}")
    print(f"    Outside India              : {outside_india:>10,}")
    print()
    print("  Recency gate:")
    print(f"    Active ≤ 14 days           : {active_14:>10,}")
    print(f"    Active 15–30 days          : {active_15_30:>10,}")
    print(f"    Active 31–60 days          : {active_31_60:>10,}")
    print(f"    Active 61–180 days         : {active_61_180:>10,}")
    print(f"    Active 181–365 days        : {active_181_365:>10,}")
    print(f"    Active > 365 days          : {active_365p:>10,}")
    print(f"    Last active null           : {active_null:>10,}")
    print()
    print("  Engagement flags:")
    print(f"    Low response rate          : {low_response:>10,}  (< 0.40 after clamping)")
    print(f"    Long notice period         : {long_notice:>10,}  (> 60 days)")
    print(f"    Not open to work           : {not_open:>10,}  (explicit false)")
    print(f"    Weak interview completion  : {weak_interview:>10,}  (< 0.50)")
    print(f"    Remote preference          : {remote_pref:>10,}")
    print()
    print("  Multiplier distribution:")
    print(f"    1.00 – 0.80  strong        : {strong:>10,}  ({strong / total:.1%})")
    print(f"    0.80 – 0.50  moderate      : {moderate:>10,}  ({moderate / total:.1%})")
    print(f"    0.50 – 0.20  weak          : {weak:>10,}  ({weak / total:.1%})")
    print(f"    0.20 – 0.05  near-floor    : {near_floor:>10,}  ({near_floor / total:.1%})")
    print(f"    0.05         floor         : {at_floor:>10,}  ({floor_pct:.1f}%)")
    print()

    if floor_pct > 10.0:
        print(f"  ⚠ WARNING: {floor_pct:.1f}% of candidates hit the floor — Step 3 may be too harsh")
    if strong / total < 0.05:
        print("  ⚠ WARNING: <5% strong availability — check if gates are too aggressive")
    if strong / total > 0.60:
        print("  ⚠ WARNING: >60% strong availability — check if gates are too lenient")

    print("=" * 76)
    print()


# ── Run Step 3 ────────────────────────────────────────────────────────────────

print(f"\nLoading data... TODAY = {TODAY}")
df = load_data()
print(f"Rows after survivor + base + redrob join: {len(df):,}")

print("\nComputing country + recency gates...")
df = compute_country_gate(df)
df = compute_recency_gate(df)

print("Computing engagement scores...")
df = compute_response_score(df)
df = compute_notice_score(df)
df = compute_intent_score(df)
df = compute_interview_score(df)
df = compute_work_mode_score(df)
df = compute_engagement_score(df)

print("Computing final multiplier + flags...")
df = compute_multiplier(df)
df = compute_flags(df)

out = df.select(OUTPUT_COLS)

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
out.write_parquet(OUTPUT_PATH)

print(f"\nWrote output → {OUTPUT_PATH}")

print_summary(out)


# ── Display useful notebook outputs ──────────────────────────────────────────

print("Output schema:")
display(out.schema)

print("\nFirst 10 rows:")
display(out.head(10).to_pandas())

print("\nTop 10 strongest availability candidates:")
display(
    out.sort("availability_multiplier", descending=True)
       .head(10)
       .to_pandas()
)

print("\nBottom 10 weakest availability candidates:")
display(
    out.sort("availability_multiplier")
       .head(10)
       .to_pandas()
)

print("\nWorked-example check: CAND_0000001")
cand_check = out.filter(pl.col("candidate_id") == "CAND_0000001")
display(cand_check.to_pandas())

if cand_check.height > 0:
    val = cand_check["availability_multiplier"][0]
    raw = cand_check["availability_raw"][0]
    print(f"CAND_0000001 availability_raw        = {raw:.6f}")
    print(f"CAND_0000001 availability_multiplier = {val:.6f}")
    print("Note: old handover expected ≈0.077 under the old harsher location/relocation gate.")
    print("This updated version should be higher because outside-India moved from 0.10 to 0.35 and relocation was removed.")
else:
    print("CAND_0000001 not found in survivors/output.")

Resolved paths:
  survivors : dataset\artifacts\survivors.parquet
  base      : dataset\artifacts\candidate_base.parquet
  redrob    : dataset\artifacts\candidate_redrob.parquet
  output    : dataset\artifacts\availability_scores.parquet

Loading data... TODAY = 2026-06-25
Rows after survivor + base + redrob join: 90,236

Computing country + recency gates...
Computing engagement scores...
Computing final multiplier + flags...

Wrote output → dataset\artifacts\availability_scores.parquet

  STEP 3 — AVAILABILITY MULTIPLIER SUMMARY
  Total processed              :     90,236

  Country gate:
    India                      :     67,795
    Country unknown/null       :          0
    Outside India              :     22,441

  Recency gate:
    Active ≤ 14 days           :          0
    Active 15–30 days          :        884
    Active 31–60 days          :     12,920
    Active 61–180 days         :     49,492
    Active 181–365 days        :     26,940
    Active > 365 days          :  

Schema([('candidate_id', String),
        ('availability_multiplier', Float64),
        ('availability_raw', Float64),
        ('country_gate', Float64),
        ('recency_gate', Float64),
        ('engagement_score', Float64),
        ('response_score', Float64),
        ('notice_score', Float64),
        ('intent_score', Float64),
        ('interview_score', Float64),
        ('work_mode_score', Float64),
        ('last_active_days_ago', Int64),
        ('response_rate_flag', Boolean),
        ('country_risk_flag', Boolean),
        ('stale_activity_flag', Boolean),
        ('long_notice_flag', Boolean),
        ('not_open_flag', Boolean),
        ('weak_interview_flag', Boolean),
        ('remote_pref_flag', Boolean)])


First 10 rows:


,candidate_id,availability_multiplier,availability_raw,country_gate,recency_gate,engagement_score,response_score,notice_score,intent_score,interview_score,work_mode_score,last_active_days_ago,response_rate_flag,country_risk_flag,stale_activity_flag,long_notice_flag,not_open_flag,weak_interview_flag,remote_pref_flag
0,CAND_0044044,0.518000,0.518000,1.00,0.70,0.740,0.65,0.85,0.65,0.55,1.0,61,True,False,False,False,True,True,False
1,CAND_0042830,0.467600,0.467600,1.00,0.70,0.668,0.65,0.55,0.65,0.90,0.8,92,True,False,False,True,True,False,True
2,CAND_0093842,0.128205,0.128205,0.35,0.45,0.814,1.00,0.85,0.65,0.55,0.8,260,False,True,True,False,True,True,True
3,CAND_0054569,0.372150,0.372150,1.00,0.45,0.827,0.65,1.00,0.65,0.90,1.0,220,True,False,True,False,True,False,False
4,CAND_0015018,0.499100,0.499100,1.00,0.70,0.713,0.65,0.70,0.65,0.90,0.8,82,True,False,False,True,True,False,True
5,CAND_0094997,0.177380,0.177380,0.35,0.70,0.724,0.85,0.55,0.65,0.90,0.8,115,False,True,False,True,True,False,True
6,CAND_0006322,0.405450,0.405450,1.00,0.45,0.901,0.85,0.85,1.00,0.90,1.0,242,False,False,True,False,False,False,False
7,CAND_0088642,0.438900,0.438900,1.00,0.70,0.627,0.30,0.70,0.65,1.00,0.8,72,True,False,False,True,True,False,True
8,CAND_0055035,0.555100,0.555100,1.00,0.70,0.793,0.85,0.70,0.65,0.90,1.0,80,False,False,False,True,True,False,False
9,CAND_0016153,0.370350,0.370350,1.00,0.45,0.823,1.00,0.70,0.65,1.00,0.8,237,False,False,True,True,True,False,True



Top 10 strongest availability candidates:


...


Bottom 10 weakest availability candidates:


,candidate_id,availability_multiplier,availability_raw,country_gate,recency_gate,engagement_score,response_score,notice_score,intent_score,interview_score,work_mode_score,last_active_days_ago,response_rate_flag,country_risk_flag,stale_activity_flag,long_notice_flag,not_open_flag,weak_interview_flag,remote_pref_flag
0,CAND_0099188,0.08316,0.08316,0.35,0.45,0.528,0.3,0.55,0.65,0.55,0.8,244,True,True,True,True,True,True,True
1,CAND_0095022,0.08316,0.08316,0.35,0.45,0.528,0.3,0.55,0.65,0.55,0.8,208,True,True,True,True,True,True,True
2,CAND_0057471,0.08316,0.08316,0.35,0.45,0.528,0.3,0.55,0.65,0.55,0.8,248,True,True,True,True,True,True,True
3,CAND_0029145,0.08316,0.08316,0.35,0.45,0.528,0.3,0.55,0.65,0.55,0.8,261,True,True,True,True,True,True,True
4,CAND_0018919,0.08316,0.08316,0.35,0.45,0.528,0.3,0.55,0.65,0.55,0.8,184,True,True,True,True,True,True,True
5,CAND_0072096,0.08316,0.08316,0.35,0.45,0.528,0.3,0.55,0.65,0.55,0.8,233,True,True,True,True,True,True,True
6,CAND_0085116,0.08316,0.08316,0.35,0.45,0.528,0.3,0.55,0.65,0.55,0.8,246,True,True,True,True,True,True,True
7,CAND_0034691,0.08316,0.08316,0.35,0.45,0.528,0.3,0.55,0.65,0.55,0.8,210,True,True,True,True,True,True,True
8,CAND_0098365,0.08316,0.08316,0.35,0.45,0.528,0.3,0.55,0.65,0.55,0.8,193,True,True,True,True,True,True,True
9,CAND_0022543,0.08316,0.08316,0.35,0.45,0.528,0.3,0.55,0.65,0.55,0.8,202,True,True,True,True,True,True,True



Worked-example check: CAND_0000001


,candidate_id,availability_multiplier,availability_raw,country_gate,recency_gate,engagement_score,response_score,notice_score,intent_score,interview_score,work_mode_score,last_active_days_ago,response_rate_flag,country_risk_flag,stale_activity_flag,long_notice_flag,not_open_flag,weak_interview_flag,remote_pref_flag
0,CAND_0000001,0.251387,0.251387,0.35,0.85,0.845,0.65,0.85,1.0,0.9,1.0,36,True,True,False,False,False,False,False


CAND_0000001 availability_raw        = 0.251387
CAND_0000001 availability_multiplier = 0.251387
Note: old handover expected ≈0.077 under the old harsher location/relocation gate.
This updated version should be higher because outside-India moved from 0.10 to 0.35 and relocation was removed.
CPU times: total: 4.41 s
Wall time: 5.2 s
